# Perfiles energéticos y efecto de la carga

Compararemos perfiles \(U(x\mid q)\) a conformaciones fijas y estudiaremos cómo una carga externa cambia la inclinación energética.

In [ ]:
from pathlib import Path
import sys

# Find the project root whether JupyterLab was launched from the repository
# root or from the notebooks/ folder.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "src" / "protein_motor_sim").is_dir()),
    None,
)
if PROJECT_ROOT is not None:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT or "package expected to be installed in this kernel")

## 1. Perfiles para diferentes coordenadas conformacionales

In [ ]:
import matplotlib.pyplot as plt
from protein_motor_sim.mechanics.energy_landscape import EnergyLandscape
from protein_motor_sim.visualization.energy_maps import plot_energy_profiles

landscape = EnergyLandscape(
    spatial_barrier_kBT=2.5,
    conformational_barrier_kBT=3.0,
    conformational_bias_kBT=0.5,
)
ax = plot_energy_profiles(
    landscape,
    x_range_nm=(-16.4, 24.6),
    conformations={"conformation A": 0.0, "intermediate": 0.5, "conformation B": 1.0},
)
plt.show()

## 2. Efecto de la fuerza externa

La contribución mecánica adimensional de la carga es \(F x/(k_B T)\), tomando \(F>0\) como fuerza que se opone al avance hacia \(+x\).

In [ ]:
import numpy as np

x = np.linspace(-8.2, 32.8, 500)
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
for force in (0.0, 1.0, 2.0, 4.0):
    energy = landscape.energy_profile_kBT(x, q=0.0, opposing_force_pN=force)
    ax.plot(x, energy, label=f"F = {force:g} pN")
ax.set_xlabel("Position x (nm)")
ax.set_ylabel(r"U / (k$_B$T)")
ax.set_title("Tilt of the energy profile under opposing load")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 3. Comprobar la inclinación

La carga añade un término lineal en \(x\). El cambio energético entre dos posiciones, por esa contribución, es \(\Delta U = F\Delta x\).

In [ ]:
force_pN = 2.0
x1_nm, x2_nm = 0.0, 8.2
u1 = landscape.energy_kBT(x1_nm, 0.0, opposing_force_pN=force_pN)
u2 = landscape.energy_kBT(x2_nm, 0.0, opposing_force_pN=force_pN)
print(f"Cambio total en U/kBT: {u2-u1:.4f}")
print(f"Contribución de carga esperada: {force_pN*(x2_nm-x1_nm)/landscape.thermal_energy_pN_nm:.4f} kBT")